# PEAK M-ATH — Dictionary DGA Detection via Supervised Classification

LSTM-based classifier for dictionary-based Domain Generation Algorithms, following the **PEAK** framework: **Prepare → Execute → Act → Knowledge**. Adapted from [Splunk PEAK dictionary_dga_classifier](https://github.com/splunk/PEAK/tree/main/dictionary_dga_classifier).

**Ref:** M06 — Dictionary DGA detection via Supervised Classification  
**M-ATH approach:** Supervised deep learning (LSTM) trained on labeled legitimate vs. dictionary-DGA domains; character-level tokenization and binary classification.

## How to use
1. Install dependencies: `pip install -r requirements-dga.txt`
2. Place domain CSVs in `input/` (must have `domain` or `event.dns.request` column)
3. Run all cells
4. Review flagged domains in `output/`

In [ ]:
pass  # Placeholder (removed environment-specific output)

## PREPARE — Plan your Approach

- **Select Topic:** Dictionary-based DGA detection — adversaries use word-list-based domain generation to evade random-character DGA detectors (MITRE ATT&CK [T1568.002](https://attack.mitre.org/techniques/T1568/002/) Dynamic Resolution: Domain Generation Algorithms).
- **Research Topic:** Deep learning for DGA classification (LSTM character-level models), Splunk PEAK dictionary_dga_classifier, labeled domain datasets.
- **Identify Datasets:** DNS logs with domain columns; pre-trained LSTM model and tokenizer from `models/`.
- **Select Algorithms:** Supervised LSTM binary classifier — character-level tokenization, TLD stripping, prediction threshold at 0.6.

In [1]:
# Scenario mode: run from scenarios/dictionary_dga_detection_via_supervised_classification/
import os
import sys
from pathlib import Path
SCENARIO_DIR = Path('.').resolve()
REPO_ROOT = SCENARIO_DIR.parent.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)
INPUT_DIR = SCENARIO_DIR / 'input'
OUTPUT_DIR = SCENARIO_DIR / 'output'
MODELS_DIR = SCENARIO_DIR / 'models'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SCENARIO_MODE = True

In [2]:
import glob
import math
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
from tqdm import tqdm
from tensorflow.keras.preprocessing import sequence

max_domain_length = 253

if not globals().get('SCENARIO_MODE', False):
    INPUT_DIR = Path('input')
    OUTPUT_DIR = Path('output')
    MODELS_DIR = Path('models')
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def _rel(p):
    try:
        if globals().get('SCENARIO_MODE', False) and 'REPO_ROOT' in globals() and hasattr(p, 'is_relative_to') and p.is_relative_to(REPO_ROOT):
            return p.relative_to(REPO_ROOT)
        return p
    except (ValueError, AttributeError):
        return p

print(f'Input folder: {_rel(INPUT_DIR)}')
print(f'Output folder: {_rel(OUTPUT_DIR)}')
print(f'Models folder: {_rel(MODELS_DIR)}')

ModuleNotFoundError: No module named 'tensorflow'

## EXECUTE — Experimentation Time

- **Gather Data:** Load domain CSVs from `input/`, detect the domain column automatically.
- **Pre-Process Data:** Normalize domains, strip TLDs, tokenize characters for LSTM input.
- **Develop Model / Apply:** Load pre-trained LSTM model and tokenizer; run binary prediction on each domain.
- **Analyze:** Flag domains with prediction above threshold (0.6), sort by confidence score.
- **Escalate Critical Findings:** Domains classified as dictionary-DGA with high confidence warrant DNS sinkholing or blocking.

## Load domain data from input/

In [ ]:
csv_paths = sorted(glob.glob(str(INPUT_DIR / '**' / '*.csv'), recursive=True))
print(f'Found {len(csv_paths)} CSV file(s).')

if not csv_paths:
    raise FileNotFoundError(f'No CSV files in {_rel(INPUT_DIR)}. Add domain data (domain or event.dns.request column).')

dfs = []
for p in csv_paths:
    df = pd.read_csv(p)
    try:
        src_rel = str(Path(p).relative_to(REPO_ROOT)) if 'REPO_ROOT' in globals() and Path(p).is_relative_to(REPO_ROOT) else p
    except (ValueError, AttributeError):
        src_rel = p
    df['_source_file'] = src_rel
    dfs.append(df)
df = pd.concat(dfs, ignore_index=True)

domain_col = next((c for c in df.columns if 'dns.request' in c.lower() or c == 'domain' or c == 'query'), None)
if not domain_col:
    raise ValueError(f'No domain column found. Columns: {list(df.columns)}')

df = df.rename(columns={domain_col: 'domain'})
df['domain'] = df['domain'].astype(str).str.strip().str.lower().str.rstrip('.')
df = df.dropna(subset=['domain'])
df = df[df['domain'].str.len() > 0]
df['orig_domain'] = df['domain'].copy()
print(f'Loaded {len(df):,} domains.')

## Load model and tokenizer

In [ ]:
model_path = MODELS_DIR / 'm-ath_dict_dga_model'
tokenizer_path = MODELS_DIR / 'm-ath_pretrain_tokenizer.pkl'

if not model_path.exists():
    raise FileNotFoundError(f'Model not found at {model_path}. Unzip m-ath_dict_dga_model.zip into models/')
if not tokenizer_path.exists():
    raise FileNotFoundError(f'Tokenizer not found at {tokenizer_path}')

model = tf.keras.models.load_model(str(model_path))
tokenizer = pickle.load(open(tokenizer_path, 'rb'))
print('Model and tokenizer loaded.')

## Preprocess and predict

In [ ]:
def preprocess(url):
    pos = url.rfind('.')
    if pos >= 0:
        url = url[:pos]
    return url

def predict_row(row):
    domain = row['domain']
    seq = tokenizer.texts_to_sequences([domain])
    X = sequence.pad_sequences(seq, maxlen=max_domain_length)
    prediction = model.predict(X, verbose=0)
    return prediction[0][0]

def round_prediction(pred):
    if pred >= 0.6:
        return 1
    return 0

In [ ]:
batch_size = 10000
num_batches = math.ceil(len(df) / batch_size)

for batch_index in range(num_batches):
    start_index = batch_index * batch_size
    end_index = min((batch_index + 1) * batch_size, len(df))
    batch_df = df.iloc[start_index:end_index]
    for index, row in tqdm(batch_df.iterrows(), total=len(batch_df), desc=f'Preprocess batch {batch_index+1}/{num_batches}'):
        df.at[index, 'domain'] = preprocess(row['domain'])

print('Preprocessing done.')

In [ ]:
df['prediction'] = df.apply(predict_row, axis=1)
df['rounded_prediction'] = df['prediction'].apply(round_prediction)
print('Predictions complete.')

## ACT — Wrapping Up the Investigation

- **Document Findings:** Flagged dictionary-DGA domains with prediction scores exported for analyst review.
- **Preserve Hunt:** Results archived to `output/dictionary_dga_results.csv`.
- **Create Detections / Playbooks:** High-confidence DGA domains can feed automated DNS blocklists or Risk-Based Alerting notables.

## Flag suspicious domains and save results

In [ ]:
flagged = df[df['rounded_prediction'] == 1][['orig_domain', 'prediction', 'rounded_prediction']].sort_values('prediction', ascending=False)
print(f'Flagged {len(flagged):,} domains as dictionary DGA.')
if len(flagged) > 0:
    display(flagged.head(20))

In [ ]:
out_path = OUTPUT_DIR / 'dictionary_dga_results.csv'
flagged.to_csv(out_path, index=False)
print(f'Saved to {_rel(out_path)}')

## KNOWLEDGE — Continuous Improvement

- **Re-Add Topics to Backlog:** New DGA families or evasion patterns discovered during analysis become retraining targets or future hunt hypotheses.
- **Communicate Findings:** Share confirmed DGA domains with DNS/firewall teams for immediate blocking and with threat intelligence for attribution.
- **Feed Improvements Back:** Retrain the LSTM with newly labeled domains (false positives as legitimate, confirmed DGA as malicious); tune the prediction threshold based on observed accuracy.
- **Measure Effectiveness:** Track detection rate, false positive rate, and newly discovered DGA families across successive runs.